In [1]:
import ee
import geemap
ee.Initialize(project='mineral-prospectivity-zim')
print('project initialized')

project initialized


In [2]:
# Load assets
feature_stack = ee.Image('projects/mineral-prospectivity-zim/assets/feature_stack')
mrds = ee.FeatureCollection('projects/mineral-prospectivity-zim/assets/mrds_zim_gold_clean')

# Zimbabwe bounding box
zim_bbox = ee.Geometry.Rectangle([25.2374, -22.4171, 33.0567, -15.6089])

# Build 15km exclusion buffer around all 61 deposit points
exclusion_buffer = mrds.geometry().buffer(15000)

# Build exclusion mask using paint() — 1 inside buffer, 0 outside, then invert
buffer_image = ee.Image.constant(0).paint(
    featureCollection=ee.FeatureCollection([ee.Feature(exclusion_buffer)]),
    color=1
)
exclusion_mask = buffer_image.Not()

In [3]:
# Select the six ND indices
s2_ioi_nd = feature_stack.select('S2_IOI_nd')
s2_fii_nd = feature_stack.select('S2_FII_nd')
s2_ci_nd  = feature_stack.select('S2_CI_nd')
l8_ioi_nd = feature_stack.select('L8_IOI_nd')
l8_fii_nd = feature_stack.select('L8_FII_nd')
l8_ci_nd  = feature_stack.select('L8_CI_nd')

# Unfavourability score — higher = more unlike mineralised ground
unfav = (
    s2_ioi_nd.multiply(-1).add(1)
    .add(s2_fii_nd.multiply(-1).add(1))
    .add(s2_ci_nd.multiply(-1).add(1))
    .add(l8_ioi_nd.multiply(-1).add(1))
    .add(l8_fii_nd.multiply(-1).add(1))
    .add(l8_ci_nd.multiply(-1).add(1))
).rename('unfav_score')

# Apply exclusion mask
unfav_masked = unfav.updateMask(exclusion_mask)

In [4]:
# Sample negatives directly — no strata, pure random sample from unmasked pixels
# outside the 15km exclusion buffer, biased toward high unfavourability

# Weight sampling by unfav score — use it as a probability band
unfav_normalized = unfav_masked.divide(6).rename('weight')

# Add weight + full feature stack, mask to exclusion zone
image_for_sampling = feature_stack.addBands(unfav_normalized).updateMask(exclusion_mask)

# Random sample — 400 points, then we'll take top 325 by unfav score on the client side
negatives_raw = image_for_sampling.sample(
    region=zim_bbox,
    scale=30,
    numPixels=2000,
    projection='EPSG:4326',
    tileScale=4,
    geometries=True,
    seed=42
)

negatives_fc = negatives_raw.map(lambda f: f.set('label', 0))

In [5]:
# unmask(0) fills cloud/no-data pixels so no deposit points are dropped
feature_stack_unmasked = feature_stack.unmask(0)

# Get band names for column selection
band_names = feature_stack.bandNames()

# Sample all 35 band values at the 61 MRDS deposit locations
positives_fc = feature_stack_unmasked.sampleRegions(
    collection=mrds,
    scale=30,
    projection='EPSG:4326',
    tileScale=4
).map(lambda f: f.set('label', 1)).select(band_names.add(ee.String('label')))

# Drop unfav_strata from negatives — keep only band columns + label
negatives_fc_clean = negatives_fc.select(band_names.add(ee.String('label')))

# Merge
training_fc = positives_fc.merge(negatives_fc_clean)

# Export to Drive
export_task = ee.batch.Export.table.toDrive(
    collection=training_fc,
    description='training_samples_v2',
    folder='mineral_prospectivity_zim',
    fileNamePrefix='training_samples_v2',
    fileFormat='CSV'
)

export_task.start()
print('Export started:', export_task.status())

Export started: {'state': 'READY', 'description': 'training_samples_v2', 'priority': 100, 'creation_timestamp_ms': 1782473211985, 'update_timestamp_ms': 1782473211985, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'E7Z7TI5P3RKA42M356MVLRSU', 'name': 'projects/mineral-prospectivity-zim/operations/E7Z7TI5P3RKA42M356MVLRSU'}
